# Fine-tune E5-base — 3 epoch (thử nghiệm)

Fine-tune **`intfloat/multilingual-e5-base`** với **3 epoch** để kiểm tra metric có tiếp tục cải thiện sau 2 epoch hay bắt đầu overfit.

| | 1 epoch | 2 epoch | Notebook này (3 epoch) |
|---|---|---|---|
| Output model | `models/e5_base_finetuned_final/` | `models/e5_base_finetuned_2ep_final/` | `models/e5_base_finetuned_3ep_final/` |
| Metrics | `metrics_e5_base.json` | `metrics_e5_base_2epochs.json` | `metrics_e5_base_3epochs.json` |

**Cần GPU Colab** (T4 trở lên khuyến nghị). Thời gian train ~3× so với 1 epoch.

> **Gợi ý:** Nếu valid loss epoch 3 cao hơn epoch 2 → dừng ở 2 epoch (`load_best_model_at_end=True` sẽ lưu checkpoint tốt nhất).

## 1) Cài thư viện

In [ ]:
!pip -q install "sentence-transformers>=3.0.0" "transformers>=4.40.0" torch datasets pandas scikit-learn numpy tqdm

## 2) Clone repo và kiểm tra GPU

In [ ]:
import os
import shutil
import subprocess
import sys
import torch

GITHUB_REPO_URL = "https://github.com/YOUR_USER/llm_provider_benchmarking.git"  # ← sửa URL
REPO_DIR = "/content/llm_provider_benchmarking"

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", "--depth", "1", GITHUB_REPO_URL, REPO_DIR], check=True)

SCRIPTS_DIR = f"{REPO_DIR}/embedding_project/scripts"
if SCRIPTS_DIR not in sys.path:
    sys.path.insert(0, SCRIPTS_DIR)

os.chdir(REPO_DIR)
print("REPO_DIR:", REPO_DIR)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 3) Cấu hình — 3 epoch

Khác bản 2 epoch: `EPOCHS = 3`, thư mục model và file metrics riêng.

In [ ]:
from pathlib import Path
from model_presets import get_preset

PRESET = get_preset("e5-base")
PROJECT_ROOT = Path(REPO_DIR) / "embedding_project"
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
OUTPUT_EVAL_DIR = PROJECT_ROOT / "outputs" / "evaluation"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_EVAL_DIR.mkdir(parents=True, exist_ok=True)

USE_GPU = torch.cuda.is_available()
EPOCHS = 3  # thử nghiệm chính
BATCH_SIZE = 8 if USE_GPU else 2
FP16 = USE_GPU
MAX_SEQ_LENGTH = PRESET.max_seq_length
LEARNING_RATE = PRESET.learning_rate
WARMUP_RATIO = PRESET.warmup_ratio

FINAL_DIR = MODELS_DIR / "e5_base_finetuned_3ep_final"
CHECKPOINT_DIR = MODELS_DIR / "e5-base-3ep"
METRICS_FILE = OUTPUT_EVAL_DIR / "metrics_e5_base_3epochs.json"
METRICS_1EP_FILE = OUTPUT_EVAL_DIR / "metrics_e5_base.json"
METRICS_2EP_FILE = OUTPUT_EVAL_DIR / "metrics_e5_base_2epochs.json"

print("Base model:", PRESET.base_model)
print("Final dir:", FINAL_DIR)
print(f"epochs={EPOCHS} | batch={BATCH_SIZE} | fp16={FP16} | max_seq={MAX_SEQ_LENGTH} | lr={LEARNING_RATE}")

## 4) Load train / valid

In [ ]:
import json
from datasets import Dataset

def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

train_rows = load_jsonl(DATA_DIR / "train_cleaned.jsonl")
valid_rows = load_jsonl(DATA_DIR / "valid_cleaned.jsonl")
print("train:", len(train_rows), "| valid:", len(valid_rows))

train_ds = Dataset.from_list([{"anchor": r["query"], "positive": r["positive"]} for r in train_rows])
valid_ds = Dataset.from_list([{"anchor": r["query"], "positive": r["positive"]} for r in valid_rows])

## 5) Fine-tune 3 epoch

Loss: `MultipleNegativesRankingLoss`. Lưu checkpoint mỗi epoch (`save_total_limit=3`).
`load_best_model_at_end=True` — nếu epoch 3 overfit, model cuối vẫn là checkpoint valid loss thấp nhất.

In [ ]:
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import SentenceTransformerTrainingArguments, BatchSamplers

model = SentenceTransformer(PRESET.base_model)
model.max_seq_length = MAX_SEQ_LENGTH
loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    fp16=FP16,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_steps=20,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    run_name="e5-base-vi-3epochs-colab",
    report_to=[],
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    loss=loss,
)

train_result = trainer.train()
model.save(str(FINAL_DIR))
print("Saved model:", FINAL_DIR)
print("Train summary:", train_result)

## 6) Loss theo epoch (train / valid)

In [ ]:
import pandas as pd

log_history = trainer.state.log_history
train_by_epoch = {}
valid_by_epoch = {}

for entry in log_history:
    ep = entry.get("epoch")
    if ep is None:
        continue
    if "loss" in entry:
        train_by_epoch[ep] = entry["loss"]
    if "eval_loss" in entry:
        valid_by_epoch[ep] = entry["eval_loss"]

epochs = sorted(set(train_by_epoch) | set(valid_by_epoch))
rows = []
for ep in epochs:
    rows.append({
        "epoch": int(ep) if float(ep).is_integer() else ep,
        "train_loss": train_by_epoch.get(ep),
        "valid_loss": valid_by_epoch.get(ep),
    })

if rows:
    df_log = pd.DataFrame(rows)
    display(df_log)
    if len(df_log) >= 2:
        last = df_log.iloc[-1]["valid_loss"]
        prev = df_log.iloc[-2]["valid_loss"]
        if last is not None and prev is not None:
            trend = "giảm" if last < prev else "tăng"
            print(f"Valid loss epoch cuối {trend} so với epoch trước ({prev:.6f} → {last:.6f})")
else:
    print("Không có log_history — xem output trainer ở cell trên.")

## 7) Đánh giá trên test (chỉ fine-tuned 3 epoch)

Dùng script `evaluate_embedding_model.py` — prefix `query:` / `passage:` cho E5.

In [ ]:
!python embedding_project/scripts/evaluate_embedding_model.py \
  --preset e5-base \
  --only-finetuned \
  --finetuned-model embedding_project/models/e5_base_finetuned_3ep_final \
  --output embedding_project/outputs/evaluation/metrics_e5_base_3epochs.json \
  --no-cache

## 8) So sánh 1 epoch vs 2 epoch vs 3 epoch

In [ ]:
import json

def load_metrics(path):
    if not path.is_file():
        return None
    return json.loads(path.read_text(encoding="utf-8"))

m1 = load_metrics(METRICS_1EP_FILE)
m2 = load_metrics(METRICS_2EP_FILE)
m3 = load_metrics(METRICS_FILE)

ft1 = (m1 or {}).get("finetuned", {})
ft2 = (m2 or {}).get("finetuned", {})
ft3 = (m3 or {}).get("finetuned", {})

metrics = ["Precision@10", "Recall@10", "MRR@10", "NDCG@10"]
rows = []
for k in metrics:
    v1, v2, v3 = ft1.get(k), ft2.get(k), ft3.get(k)
    delta_3v2 = "—"
    if v2 is not None and v3 is not None and v2 != 0:
        delta_3v2 = f"{(v3 - v2) / v2 * 100:+.1f}%"
    rows.append({
        "Metric": k,
        "1 epoch": v1,
        "2 epoch": v2,
        "3 epoch": v3,
        "Δ (3 vs 2)": delta_3v2,
    })

df_cmp = pd.DataFrame(rows)
display(df_cmp)

missing = []
if not ft1:
    missing.append("metrics_e5_base.json (1 epoch)")
if not ft2:
    missing.append("metrics_e5_base_2epochs.json (2 epoch)")
if missing:
    print("Thiếu file so sánh:", ", ".join(missing))
    print("Upload vào outputs/evaluation/ hoặc chạy eval các bản trước.")

if ft2 and ft3:
    better = sum(1 for k in metrics if ft3.get(k, 0) > ft2.get(k, 0))
    worse = sum(1 for k in metrics if ft3.get(k, 0) < ft2.get(k, 0))
    print(f"3 epoch tốt hơn 2 epoch trên {better}/{len(metrics)} metric, kém hơn {worse}/{len(metrics)}.")
    if better == 0 and worse > 0:
        print("→ Gợi ý: dừng ở 2 epoch, có thể đã overfit.")
    elif better >= 3:
        print("→ Gợi ý: 3 epoch có lợi, cân nhắc index Qdrant collection mới.")

## 9) Tải model về máy (zip)

In [ ]:
import shutil
from google.colab import files

assert FINAL_DIR.is_dir(), f"Chưa có model: {FINAL_DIR}"

zip_path = shutil.make_archive("/content/e5_base_finetuned_3ep_final", "zip", root_dir=str(FINAL_DIR))
print("Zip:", zip_path)
files.download(zip_path)

## 10) (Tùy chọn) Tải metrics JSON

In [ ]:
from google.colab import files

if METRICS_FILE.is_file():
    files.download(str(METRICS_FILE))
else:
    print("Chưa có file metrics — chạy cell đánh giá trước.")

## 11) (Tùy chọn) Index Qdrant sau khi tải model về máy

Giải nén vào `embedding_project/models/e5_base_finetuned_3ep_final/`, rồi chạy local:

```bash
python vector_db/03_index_to_qdrant.py --recreate \
  --collection products_vi_e5_3ep \
  --model-path embedding_project/models/e5_base_finetuned_3ep_final \
  --e5-prefix --encode-batch-size 8

python vector_db/04_search_test.py --collection products_vi_e5_3ep \
  --model-path embedding_project/models/e5_base_finetuned_3ep_final \
  --e5-prefix --query "giày chạy bộ nam" --top-k 5
```